# Highway Networks on CIFAR-10

A minimal, pedagogical PyTorch implementation of
[Highway Networks](https://arxiv.org/abs/1505.00387) (Srivastava, Greff, Schmidhuber, 2015),
trained on CIFAR-10.

The notebook trains two networks that are identical except for their hidden blocks:

* a **plain** deep MLP (`Linear -> ReLU`), and
* a **highway** MLP (`Highway` blocks with learned transform gates).

Comparing them illustrates what the gates do: a deep network can learn, layer by layer and unit
by unit, how much computation to apply and how much of the input to simply carry forward.

**Running it.** A GPU is recommended but not required. On Colab choose
*Runtime > Change runtime type > GPU*; `torch` and `torchvision` are already installed there.
Locally, run `uv sync` first and select the project kernel. CIFAR-10 is downloaded to `./data`
on the first run and takes about a minute to train on a modern GPU.

## The idea in one equation

A highway layer computes

$$
y = H(x) \odot T(x) + x \odot C(x),
\qquad T(x) = \sigma(W_T x + b_T), \qquad C(x) = 1 - T(x).
$$

* $H$ is the usual nonlinear transform (here `Linear -> ReLU`).
* $T$ is the **transform gate**: a sigmoid, so every unit gets its own value in $(0, 1)$.
* $C$ is the **carry gate**: the fraction of the input that passes through unchanged.
* Input and output must have the same shape, because the shortcut is added elementwise.

The gates are *data-dependent*. If $T \approx 1$ the layer behaves like an ordinary layer; if
$T \approx 0$ it is (almost) the identity. A stack of highway layers can therefore adapt its
effective depth to the input.

The one crucial implementation detail is the initialization of the gate bias: it is set
**negative** (here $-2$), so gates start mostly closed and the deep network initially behaves
like a shallow one. Layers then open up during training as they become useful. This is what made
networks with hundreds of layers trainable before normalization and residual connections became
standard.

For contrast, a residual connection computes $y = H(x) + x$: the identity path is unconditional.
A highway layer learns a per-unit interpolation between $H(x)$ and $x$.

## Setup

In [1]:
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

torch 2.15.0+cu132 | CUDA available: True
NVIDIA GeForce RTX 3060 Ti


In [2]:
SEED = 0
DATA_ROOT = Path("data")
BATCH_SIZE = 256
EPOCHS = 15
LR = 1e-3
WIDTH = 512
DEPTH = 12  # hidden blocks after the stem
NUM_WORKERS = 2  # set to 0 on Windows if the data loader hangs


def set_seed(seed=SEED):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## Data

CIFAR-10 has 50,000 training and 10,000 test images of size $32 \times 32$, in 10 classes.
Training images get random crops and horizontal flips; both splits are normalized with the
dataset mean and standard deviation.

In [3]:
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)

train_tf = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ]
)
test_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

train_set = datasets.CIFAR10(DATA_ROOT, train=True, download=True, transform=train_tf)
test_set = datasets.CIFAR10(DATA_ROOT, train=False, download=True, transform=test_tf)

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
)
test_loader = DataLoader(
    test_set,
    batch_size=512,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
)

print(f"{len(train_set):,} train / {len(test_set):,} test images")

100.0%


50,000 train / 10,000 test images


## The highway layer

Two linear maps of the same shape, one producing $H$ and one producing $T$, followed by the
convex combination. The mean gate value is cached during the forward pass so we can inspect what
the trained layers learned.

In [4]:
class Highway(nn.Module):
    """y = H(x) * T(x) + x * (1 - T(x)), with T = sigmoid(gate(x))."""

    def __init__(self, dim: int, gate_bias: float = -2.0):
        super().__init__()
        self.transform = nn.Linear(dim, dim)
        self.gate = nn.Linear(dim, dim)
        # Negative bias: start with mostly-closed gates, i.e. close to the identity.
        nn.init.constant_(self.gate.bias, gate_bias)
        self.last_gate = None

    def forward(self, x):
        h = F.relu(self.transform(x))
        t = torch.sigmoid(self.gate(x))
        self.last_gate = t.detach().mean()
        return h * t + x * (1 - t)

## Plain vs. highway network

Both models share the same stem (`Flatten -> Linear -> ReLU`), the same number of hidden blocks,
the same width, and the same classifier. Only the block type differs.

One caveat: a highway block has two linear maps, so the highway model has roughly twice as many
parameters as the plain one. The comparison is about trainability at depth, not about an
iso-parameter budget. (An exercise below is to widen the plain net to match.)

In [5]:
class Net(nn.Module):
    def __init__(self, depth: int = DEPTH, width: int = WIDTH, highway: bool = True):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * 32 * 32, width),
            nn.ReLU(),
        )
        if highway:
            self.blocks = nn.ModuleList(Highway(width) for _ in range(depth))
        else:
            self.blocks = nn.ModuleList(
                nn.Sequential(nn.Linear(width, width), nn.ReLU()) for _ in range(depth)
            )
        self.head = nn.Linear(width, 10)

    def forward(self, x):
        x = self.stem(x)
        for block in self.blocks:
            x = block(x)
        return self.head(x)


def n_params(model):
    return sum(p.numel() for p in model.parameters())

## Training loop

Standard classification loop: Adam with a cosine learning-rate schedule, cross-entropy loss.
Test accuracy is measured after every epoch.

In [6]:
@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(dim=1) == y).sum().item()
        total += y.numel()
    return correct / total


def train(model, epochs: int = EPOCHS, lr: float = LR):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.CrossEntropyLoss()

    history = {"loss": [], "acc": []}
    start = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(x), y)
            loss.backward()
            opt.step()
            running += loss.item() * y.numel()
        sched.step()

        history["loss"].append(running / len(train_set))
        history["acc"].append(accuracy(model, test_loader))
        print(
            f"epoch {epoch:2d}/{epochs} | loss {history['loss'][-1]:.3f} "
            f"| test acc {history['acc'][-1]:.1%}"
        )

    print(f"trained in {time.time() - start:.1f}s")
    return history

## Train both models

The same seed is reset before each run, so initialization and data order are comparable.

In [ ]:
set_seed()
plain = Net(highway=False)
print(f"plain MLP:   {n_params(plain):,} parameters")
plain_hist = train(plain)

plain MLP:   4,730,378 parameters
epoch  1/15 | loss 2.075 | test acc 21.7%
epoch  2/15 | loss 1.984 | test acc 24.0%
epoch  3/15 | loss 1.938 | test acc 25.4%


In [ ]:
set_seed()
highway = Net(highway=True)
print(f"highway MLP: {n_params(highway):,} parameters")
highway_hist = train(highway)

## Results

In [ ]:
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(10, 3.5))

for name, hist in [("plain", plain_hist), ("highway", highway_hist)]:
    ax_loss.plot(hist["loss"], label=name)
    ax_acc.plot([100 * a for a in hist["acc"]], label=name)

ax_loss.set(xlabel="epoch", ylabel="train loss", title="Training loss")
ax_acc.set(xlabel="epoch", ylabel="test accuracy (%)", title="Test accuracy")
for ax in (ax_loss, ax_acc):
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

for name, hist in [("plain", plain_hist), ("highway", highway_hist)]:
    print(f"{name:8s} final test accuracy: {hist['acc'][-1]:.1%}")

## What did the gates learn?

Average transform-gate value per layer on the test set. Values near 0 mean the layer mostly
copies its input (carry); values near 1 mean it mostly transforms it. The profile shows how the
network allocates its depth.

In [ ]:
@torch.no_grad()
def gate_means(model, loader):
    model.eval()
    sums = torch.zeros(len(model.blocks))
    for x, _ in loader:
        model(x.to(device))
        sums += torch.tensor([block.last_gate.item() for block in model.blocks])
    return (sums / len(loader)).tolist()


means = gate_means(highway, test_loader)

plt.figure(figsize=(8, 3))
plt.bar(range(1, len(means) + 1), means, color="tab:orange")
plt.axhline(0.5, color="gray", linestyle="--", linewidth=1)
plt.xlabel("highway layer")
plt.ylabel("mean transform gate T")
plt.ylim(0, 1)
plt.title("How much each highway layer transforms its input")
plt.tight_layout()
plt.show()

## A look at predictions

In [ ]:
classes = train_set.classes
images, labels = next(iter(test_loader))
images, labels = images[:8], labels[:8]  # one batch holds 512, show 8
with torch.no_grad():
    probs = highway(images.to(device)).softmax(dim=1).cpu()
labels = labels.tolist()
preds = probs.argmax(dim=1).tolist()
confs = probs.max(dim=1).values.tolist()

fig, axes = plt.subplots(2, 4, figsize=(10, 5.5))
for ax, img, label, pred, conf in zip(axes.flat, images, labels, preds, confs, strict=True):
    rgb = (img * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]).clamp(0, 1)
    ax.imshow(rgb.permute(1, 2, 0))
    ax.set_title(
        f"true: {classes[label]}\npred: {classes[pred]} ({conf:.0%})",
        fontsize=9,
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

## Takeaways and things to try

* A highway layer is a tiny change: two linear maps and a sigmoid. The learned gate decides,
  per unit, whether to transform or to carry the input forward.
* Negative gate-bias initialization matters. It starts the deep network near the identity and
  lets it grow into depth instead of having to train a deep plain stack from scratch.
* This notebook uses raw pixels, so absolute accuracies are modest (a 2-layer MLP is already a
  strong baseline on CIFAR-10). Replace the stem with a small convolutional stack and both
  models should improve; does the highway advantage remain?
* If the plain net trains just as well at depth 12, increase `DEPTH`. The gap between plain and
  highway stacks usually widens with depth, which is exactly the regime the paper targets.
* The parameter counts are not matched. Widen the plain model to match the highway model.
* Sweep `DEPTH` (4, 12, 24) and `gate_bias` (-4, -2, 0). Plot final accuracy against depth.
  The paper's claim is that highway layers keep very deep networks trainable.
* Replace `Highway` with a residual block ($y = H(x) + x$). When is a learned gate better than
  an unconditional identity path?
* In the paper, $H$ can also be a convolution, a recurrent layer, or anything shape-preserving.
  The gate machinery stays the same.